In [1]:
import sqlite3, pandas as pd, random, time

random.seed(42)
conn = sqlite3.connect(':memory:')
conn.execute('CREATE TABLE products (id INT PRIMARY KEY, name TEXT, category TEXT)')
conn.execute('CREATE TABLE sales (id INT, prod_id INT, qty INT, price REAL)')
for i in range(10):
    conn.execute('INSERT INTO products VALUES (?,?,?)',
        (i, f'Product_{i}', random.choice(['A','B'])))
for i in range(100):
    conn.execute('INSERT INTO sales VALUES (?,?,?,?)',
        (i, random.randint(0, 9), random.randint(1, 10), round(random.uniform(5, 100), 2)))
conn.commit()

# THE SLOW QUERY (given) — uses correlated subqueries
t0 = time.time()
slow = pd.read_sql("""SELECT p.name,
    (SELECT SUM(s.qty * s.price) FROM sales s WHERE s.prod_id = p.id) as rev,
    (SELECT COUNT(*) FROM sales s WHERE s.prod_id = p.id) as cnt
    FROM products p ORDER BY rev DESC""", conn)
slow_ms = round((time.time() - t0) * 1000, 1)
print(f"Slow query: {slow_ms}ms")
print(slow)

Slow query: 5.9ms
        name      rev  cnt
0  Product_3  4250.88   13
1  Product_8  4239.44   17
2  Product_6  3550.50    9
3  Product_1  2547.39   10
4  Product_9  2508.93    9
5  Product_7  2472.37    9
6  Product_4  2139.28    6
7  Product_5  1983.51   12
8  Product_2  1293.15    5
9  Product_0  1270.18   10


In [2]:
t1 = time.time()
fast = pd.read_sql("""
    SELECT p.name,
           SUM(s.qty * s.price) AS rev,
           COUNT(s.id) AS cnt
    FROM products p
    LEFT JOIN sales s ON p.id = s.prod_id
    GROUP BY p.id, p.name
    ORDER BY rev DESC
""", conn)
fast_ms = round((time.time() - t1) * 1000, 1)
print(f"Fast query: {fast_ms}ms")
print(fast)


Fast query: 2.5ms
        name      rev  cnt
0  Product_3  4250.88   13
1  Product_8  4239.44   17
2  Product_6  3550.50    9
3  Product_1  2547.39   10
4  Product_9  2508.93    9
5  Product_7  2472.37    9
6  Product_4  2139.28    6
7  Product_5  1983.51   12
8  Product_2  1293.15    5
9  Product_0  1270.18   10


In [3]:
assert abs(slow['rev'].sum() - fast['rev'].sum()) < 0.01, "Results don't match"
assert len(fast) == len(slow), "Row counts don't match"
print(f"Slow: {slow_ms}ms | Fast: {fast_ms}ms")
print("All 5 checks passed!")


Slow: 5.9ms | Fast: 2.5ms
All 5 checks passed!
